In [46]:
#dependencies
%pip install -q pypdf scikit-learn pandas numpy matplotlib seaborn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [47]:
from pypdf import PdfReader
import re, json
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 90)
pd.set_option("display.width", 200)

print("Core setup complete.")

Core setup complete.


In [48]:
## load the pdf file
NPR_PATH = "N_PR_7150_002D_.pdf"
reader = PdfReader(NPR_PATH)

In [49]:
## extract text from the PDF and count the number of pages and characters
raw = "\n".join((p.extract_text() or "") for p in reader.pages)
print(f"Pages: {len(reader.pages)} Characters: {len(raw):,}")

Pages: 89 Characters: 212,399


In [50]:
source_urls = {
    "2": "https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter2",
    "3": "https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter3",
    "4": "https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter4",
    "5": "https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter5"
}

In [51]:
print(reader.pages[7].extract_text())

Chapter 1: Introduction
1.1 Overview
1.1.1 This directive imposes requirements on procedures, design considerations, activities, and tasks
used to acquire, develop, maintain, operate, retire, and manage applicable software. This directive is
a designed set of requirements for protecting the ’Agency’s investment in software engineering
products and fulfilling our responsibility to the citizens of the United States (U.S.). 
1.1.2 The requirements in this directive have been extracted from industry standards and proven
NASA experience in software engineering. Centers and software developers may show that many of
the requirements are satisfied through existing programs, procedures, and processes. 
1.1.3 The Agency makes significant investments in software engineering to support the Agency’s
investment areas: Space Flight, Aeronautics, Research and Technology, Information Technology
(IT), and Institutional Infrastructure. NASA ensures that programs, projects, systems, and
subsystems that us

In [52]:
##look for the page header beginning with 'NPR 7150...' and ending with 'Page X of 89' and remove it from the text
text = re.sub(r"NPR 7150\.2D --.*?Page \d+ of 89", " ", raw, flags=re.S)

In [53]:
##find the footer "This document does not bind the public..." and remove it from the text
text = re.sub(r"This document does not bind the public.*?nodis3\.gsfc\.nasa\.gov\.", " ", text, flags=re.S)

In [54]:
## replace multiple whitespace characters with a single space and print the number of characters
text = re.sub(r"\s+", " ", text)
print(f"After cleaning: {len(text):,} characters")

After cleaning: 183,947 characters


In [55]:
#Backward-window parse: for each [SWE-###] tag, walk back to the nearest section heading.
SECTION = re.compile(r"(?:^|\s)(\d+(?:\.\d+)+)\s+(?=[A-Za-z\u201c\"(])")
SWE_TAG = re.compile(r"\[SWE-(\d{3})\]")

clauses, seen, cursor = [], set(), 0
for m in SWE_TAG.finditer(text):
    swe = m.group(1)
    if swe in seen:
        continue
    seen.add(swe)
    window = text[cursor:m.start()]
    hits = list(SECTION.finditer(window))
    if hits:
        section, body = hits[-1].group(1), window[hits[-1].end():]
    else:
        section, body = None, window[-600:]
    clauses.append({"swe_id": f"SWE-{swe}", "section": section, "text": body.strip()})
    cursor = m.end()

In [56]:
#Create a DataFrame from the clauses and add a source column
df_clauses = pd.DataFrame(clauses)
df_clauses['source'] = "NPR 7150.2D"

In [57]:
#Add source url to the dataframe
df_clauses['source_url'] = (df_clauses['section'].str.split('.').str[0].map(source_urls))

In [58]:
##display the first 20 rows of the dataframe
df_clauses.head(20)

,swe_id,section,text,source,source_url
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA So...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the pro...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals agains...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
5,SWE-098,2.1.1.6,The NASA OCE shall maintain an Agency-wide pro...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
6,SWE-208,2.1.2.2,"The NASA Chief, SMA shall lead and maintain a ...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
7,SWE-209,2.1.2.3,"The NASA Chief, SMA shall periodically benchma...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
8,SWE-212,2.1.2.4,"The NASA Chief, SMA shall periodically review ...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
9,SWE-221,2.1.2.5,"The NASA Chief, SMA shall authorize appraisals...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...


In [59]:
# Load SRS file
SRS_PATH = "5.09 - SRS - Software Requirements Specification - SW Engineering Handbook Ver D - Global Site.pdf"
reader1 = PdfReader(SRS_PATH)

In [60]:
# Extract text from the SRS PDF and count the number of pages and characters
raw1 = "\n".join((p.extract_text() or "") for p in reader1.pages)
print(f"Pages: {len(reader1.pages)} Characters: {len(raw1):,}")

Pages: 26 Characters: 102,815


In [61]:
print(reader1.pages[25].extract_text())

Examples:
It is assumed that temperature sensors will provide accurate readings within ±2°C.
It is assumed that operators are trained to manually override automated controls in critical situations.
Software is developed under the assumption of nominal solar radiation levels during operation.
3.7  SRS Best Practices
To comply with modern standards and effectively convey requirements:
1. Use Clear and Precise Language: Write measurable, testable, and unambiguous requirements, avoiding vague terms like "efﬁcient" or "easy to use."
2. Align with Industry Standards: Follow frameworks such as ISO/IEC/IEEE 29148:2011 or similar for software requirement deﬁnitions.
3. Visual Aids and Models: Enhance understanding using diagrams (e.g., UML models, data ﬂow diagrams, or state machines) and prototyping where applicable.
4. Enable Traceability: Maintain a requirements traceability matrix (RTM) to map requirements to design, code, and testing activities. Include system hazards, if applicable.
5. Su

In [62]:
# Remove section 4
text1 = re.sub(r"4. Small Projects[\s\S]*$", " ", raw1, flags=re.S)

In [63]:
## replace multiple whitespace characters with a single space and print the number of characters
text1 = re.sub(r"\s+", " ", text1)
print(f"After cleaning: {len(text1):,} characters")

After cleaning: 99,881 characters


In [64]:
import re
import pandas as pd

HEADING = re.compile(r"(?:(?<=\s)|^)(\d+(?:\.\d+)+)\s+(?=[A-Z])")
matches = list(HEADING.finditer(text1))
print(f"headings found: {len(matches)}")

sections = []
for i, m in enumerate(matches):
    section = m.group(1)
    start   = m.end()
    end     = matches[i + 1].start() if i + 1 < len(matches) else len(text1)
    text    = text1[start:end].strip()

    sections.append({
        "section": section,
        "text": text
    })

df_sections = pd.DataFrame(sections)
df_sections['source'] = "SWEHB 5.09"
print(df_sections.shape)

headings found: 46
(46, 3)


In [65]:
#add the source url to the dataframe
df_sections['source_url'] = "https://swehb.nasa.gov/spaces/SWEHBVD/pages/102695669/5.09+-+SRS+-+Software+Requirements+Specification"

In [66]:
#display the first 10 rows of the dataframe
df_sections.head(10)

,section,text,source,source_url
0,3.1,Introduction The introductory section provides...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
1,3.1.1,Purpose Specify the purpose of the SRS and how...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
2,3.1.2,Scope Describe the software product under deve...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
3,3.1.3,Software System Overview Provide an overview o...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
4,3.2,"CSCI Requirements For purposes including, but ...",SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
5,3.2.1,Functional Requirements Functional requirement...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
6,3.2.1.1,General Functional Requirements The general fu...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
7,3.2.1.1.1,Decomposed Requirements Decomposed requirement...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
8,3.2.1.1.2,Derived Requirements Derived requirements are ...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
9,3.2.1.1.3,Summary of Relationships Type Origin Purpose T...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...


In [67]:
df_result = pd.concat([df_clauses, df_sections], axis=0, ignore_index=True)
df_result.head(200)

,swe_id,section,text,source,source_url
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA So...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the pro...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals agains...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?In...
...,...,...,...,...,...
171,NaN,3.4,Rationale and Supporting Information Supportin...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
172,NaN,3.5,Additional Requirements and Information In ord...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
173,NaN,3.6,Assumptions and Limitations It is best to obta...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...
174,NaN,3.7,SRS Best Practices To comply with modern stand...,SWEHB 5.09,https://swehb.nasa.gov/spaces/SWEHBVD/pages/10...


In [ ]:
json_string = df_result.to_json(orient='records', indent=4)
print(json_string[:1000])  # Print the first 1000 characters of the JSON string

[
    {
        "swe_id":"SWE-002",
        "section":"2.1.1.1",
        "text":"The NASA OCE shall lead and maintain a NASA Software Engineering Initiative to advance software engineering practices.",
        "source":"NPR 7150.2D",
        "source_url":"https:\/\/nodis3.gsfc.nasa.gov\/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter2"
    },
    {
        "swe_id":"SWE-004",
        "section":"2.1.1.2",
        "text":"The NASA OCE shall periodically benchmark each Center\u2019s software engineering capability against requirements in this directive.",
        "source":"NPR 7150.2D",
        "source_url":"https:\/\/nodis3.gsfc.nasa.gov\/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter2"
    },
    {
        "swe_id":"SWE-152",
        "section":"2.1.1.3",
        "text":"The NASA OCE shall periodically review the project requirements mapping matrices.",
        "source":"NPR 7150.2D",
        "source_url":"https:\/\/nodis3.gsfc.nasa.gov\/displayDir.cfm?Internal